# Label alignment

The cell-type probe's labels come from **CellViT** (a deep-learning nuclear segmentation
of the H&E, carrying a PanNuke morphological `class`), transferred onto the **Xenium**
cells we embedded. CellViT and Xenium segment the same tissue *independently*, so there is
no shared `cell_id` -- the transfer is **spatial**: for each Xenium cell, the majority
CellViT class among the nucleus centroids that fall inside its polygon (nearest-centroid
fallback otherwise).

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

# Import the cellbridge package straight from src/ (no install), like tests/conftest.py.
PROJECT_ROOT = Path("/lustre/scratch126/cellgen/lotfollahi/so25/projects/cellbridge")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from cellbridge.helpers.labels import (  # noqa: E402
    PX_UM,
    load_cell_polygons,
    load_cellvit_points,
    transfer_labels,
)
from cellbridge.exploration.run_celltype_probe import load_labels  # noqa: E402
from cellbridge.helpers.io import load_embeddings  # noqa: E402

cfg = yaml.safe_load(open(PROJECT_ROOT / "configs" / "exploration" / "celltype.yaml"))
HEST_DIR = Path("/lustre/scratch126/cellgen/lotfollahi/so25/data/hest_samples")
SAMPLE_ID = cfg["sample_id"]

# One consistent color per CellViT class, reused in every figure.
CLASS_COLORS = {
    "Neoplastic": "#d62728",  # tumor
    "Connective": "#1f77b4",  # stroma
    "Inflammatory": "#2ca02c",  # immune
    "Epithelial": "#ff7f0e",
    "Unassigned": "#7f7f7f",
}
plt.rcParams["figure.dpi"] = 110
print("sample:", SAMPLE_ID, "| micron/px:", PX_UM)

In [ ]:
_, cell_ids = load_embeddings(cfg["terra_embeddings"], key=cfg["terra_key"])
terra_npz = np.load(cfg["terra_embeddings"], allow_pickle=True)
he = np.c_[terra_npz["he_x"], terra_npz["he_y"]].astype(
    float
)  # H&E level-0 px centroids

print(f"{len(cell_ids):,} embedded cells")
print(f"he_x range [{he[:, 0].min():.0f}, {he[:, 0].max():.0f}] px")
print(f"he_y range [{he[:, 1].min():.0f}, {he[:, 1].max():.0f}] px")

## Segmentations

- **Xenium** native cell polygons (`xenium_seg`), keyed by Xenium `cell_id` -- our 10k are a subset.
- **CellViT** nuclei with a morphological `class` (`cellvit_seg`), keyed by CellViT's own index.

In [ ]:
xen = load_cell_polygons(cfg["xenium_cell_seg"], cell_ids)  # our cells' Xenium polygons
cv_pts = load_cellvit_points(cfg["cellvit_seg"])  # ALL CellViT nuclei: centroid + class

print(f"Xenium cell polygons (our cells): {len(xen):,}")
print(f"CellViT nuclei (all):             {len(cv_pts):,}\n")
print(cv_pts["cellvit_class"].value_counts().to_string())

## Transfer

`transfer_labels` is the exact function the pipeline calls: polygon-containment majority
vote, with a nearest-centroid fallback beyond which cells are `Unassigned`. It returns one
row per embedded cell, recording the assigned class, how many CellViT nuclei were inside
(`n_nuclei`), the method used, and the fallback distance.

In [ ]:
labels = transfer_labels(
    cell_ids, he, xen, cv_pts, fallback_max_um=cfg["fallback_max_um"]
)
ct = labels["cell_type"].to_numpy()

print("=== class distribution ===")
print(labels["cell_type"].value_counts(dropna=False).to_string())
print("\n=== assignment method ===")
print(labels["method"].value_counts().to_string())
labels.head()

## Do the two segmentations register on the H&E?

Load the *full* Xenium cell segmentation (all ~893k polygons; a few seconds; ~1 GB) so a
crop shows real tissue density, pick a window with a tumor/stroma mix, and overlay both
segmentations on the H&E.

In [ ]:
import geopandas as gpd
import tifffile
import zarr

xen_all = gpd.read_parquet(cfg["xenium_cell_seg"])  # all Xenium cells (index = cell_id)
wsi = zarr.open(
    tifffile.imread(HEST_DIR / "wsis" / f"{SAMPLE_ID}.tif", aszarr=True, level=0),
    mode="r",
)
H, W = wsi.shape[:2]
print(f"all Xenium cells: {len(xen_all):,} | WSI level-0: {W} x {H} px")

# Pick the ~180 um window (>=15 of our cells) with the most balanced tumor/stroma mix.
win_px = 180 / PX_UM
gx, gy = (he[:, 0] // win_px).astype(int), (he[:, 1] // win_px).astype(int)
grid = pd.DataFrame({"gx": gx, "gy": gy, "ct": ct})
scores = grid.groupby(["gx", "gy"])["ct"].apply(
    lambda s: min((s == "Neoplastic").sum(), (s == "Connective").sum())
    if len(s) >= 15
    else -1
)
bx, by = scores.idxmax()
cx, cy = (bx + 0.5) * win_px, (by + 0.5) * win_px
x0, y0 = max(int(cx - win_px), 0), max(int(cy - win_px), 0)  # 2x window for context
x1, y1 = min(int(cx + win_px), W), min(int(cy + win_px), H)
patch = wsi[y0:y1, x0:x1, :3]
print(
    f"crop x[{x0},{x1}] y[{y0},{y1}] ~ {(x1 - x0) * PX_UM:.0f} x {(y1 - y0) * PX_UM:.0f} um"
)

In [ ]:
xen_win = xen_all.cx[x0:x1, y0:y1]  # Xenium cells overlapping the crop
cvx, cvy = cv_pts.geometry.x.to_numpy(), cv_pts.geometry.y.to_numpy()
in_win = (cvx >= x0) & (cvx < x1) & (cvy >= y0) & (cvy < y1)
our = (he[:, 0] >= x0) & (he[:, 0] < x1) & (he[:, 1] >= y0) & (he[:, 1] < y1)

fig, axes = plt.subplots(1, 2, figsize=(15, 7.5))
for ax in axes:
    ax.imshow(patch)
    ax.set_xticks([])
    ax.set_yticks([])

# Left: the two segmentations. Xenium cell outlines + CellViT nucleus centroids by class.
xen_win.translate(-x0, -y0).boundary.plot(ax=axes[0], color="yellow", linewidth=0.5)
for cls, col in CLASS_COLORS.items():
    m = in_win & (cv_pts["cellvit_class"].to_numpy() == cls)
    if m.any():
        axes[0].scatter(
            cvx[m] - x0,
            cvy[m] - y0,
            s=12,
            c=col,
            label=cls,
            edgecolor="k",
            linewidth=0.2,
        )
axes[0].legend(markerscale=1.5, fontsize=7, loc="upper right", framealpha=0.9)
axes[0].set_title("Xenium cells (yellow) + CellViT nucleus centroids (by class)")

# Right: the transferred label on our embedded cells (the classifier's targets).
xen_win.translate(-x0, -y0).boundary.plot(
    ax=axes[1], color="white", linewidth=0.3, alpha=0.5
)
for cls, col in CLASS_COLORS.items():
    m = our & (ct == cls)
    if m.any():
        axes[1].scatter(
            he[m, 0] - x0,
            he[m, 1] - y0,
            s=55,
            c=col,
            label=cls,
            edgecolor="k",
            linewidth=0.5,
        )
axes[1].legend(markerscale=1, fontsize=7, loc="upper right", framealpha=0.9)
axes[1].set_title("Our embedded cells, colored by transferred class")
fig.suptitle(f"{SAMPLE_ID}: Xenium / CellViT overlap and the resulting labels", y=1.02)
fig.tight_layout()

## Transfer diagnostics

How clean was the assignment? We look at (a) the nearest-nucleus distance for the fallback
cells (vs the `fallback_max_um` cutoff), (b) how many CellViT nuclei landed inside each
Xenium cell -- 1 is a clean 1:1 match, 0 means fallback, >1 means a merged/ambiguous cell --
and (c) the final class distribution.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

fb = labels.loc[labels["method"] == "fallback", "fallback_dist_um"].dropna()
axes[0].hist(fb, bins=40, color="slategray")
axes[0].axvline(
    cfg["fallback_max_um"],
    color="red",
    ls="--",
    label=f"{cfg['fallback_max_um']} um cutoff",
)
axes[0].set_title(f"Fallback cells: nearest-nucleus dist (n={len(fb):,})")
axes[0].set_xlabel("micron")
axes[0].legend()

poly_n = labels.loc[labels["method"] == "polygon", "n_nuclei"]
axes[1].hist(poly_n, bins=np.arange(0, int(poly_n.max()) + 2) - 0.5, color="teal")
axes[1].set_title("CellViT nuclei inside each Xenium cell")
axes[1].set_xlabel("# nuclei inside (polygon-matched cells)")

vc = labels["cell_type"].value_counts()
axes[2].bar(vc.index, vc.values, color=[CLASS_COLORS.get(c, "gray") for c in vc.index])
axes[2].set_title("Transferred class (all 10k)")
axes[2].tick_params(axis="x", rotation=45)
fig.tight_layout()

print(labels["method"].value_counts().to_string())
print(
    f"\nmedian fallback distance: {fb.median():.2f} um; "
    f"{100 * (fb <= cfg['fallback_max_um']).mean():.1f}% of fallbacks within cutoff"
)